# P1-E08.1 — Leakage-Free CAUEEG  Benchmark Lock

**Paper 1, Experiment E08.successor to the passed P1-E07 full-cohort preprocessing lock**

## Goal

Construct reproducible conventional EEG representations and establish leakage-free non-spiking benchmarks for:

1. Normal vs MCI vs AD;
2. Normal vs AD; and
3. MCI vs AD.

The notebook authenticates the 914 P1-E07. eligible recording arrays, extracts features without diagnosis labels, aggregates epochs to one recording-level vector, and compares spectral, temporal and hybrid representations. Model normalization and hyperparameter tuning occur inside training-only cross-validation. Candidate selection uses the official validation split. The official test split is evaluated once after the representation, model family and hyperparameters are frozen.



Run all cells in order. Continue only if the final status is:

> **P1-E08. PASSED — BENCHMARK LOCKED**

## Context & Methods

### Representations

For channel (c), epoch (e), and band (b), absolute and relative band powers are

\[
P_{e,c,b}=\int_{f\in b}S_{e,c}(f)df,\qquad
R_{e,c,b}=\frac{P_{e,c,b}}{\int_{1}^{40}S_{e,c}(f)df}.
\]

Bands are delta (1–4 Hz), theta (4–8 Hz), alpha (8–13 Hz), beta (13–30 Hz), and low gamma (30–40 Hz). Additional spectral features are spectral entropy, 95% spectral-edge frequency, and alpha-peak frequency. Temporal features are Hjorth activity, mobility and complexity plus line length.

Each epoch feature is aggregated within recording using the median and interquartile range. Thus every recording contributes exactly one independent feature vector regardless of whether it contains 6 or 12 retained epochs.

### Leakage controls

- Feature extraction is deterministic and diagnosis-independent.
- Standardization is fitted inside each training fold.
- Hyperparameters are selected with stratified five-fold cross-validation on the official training split.
- The official validation split selects the final representation/model candidate.
- The selected candidate is refitted on training + validation.
- The official test split is evaluated once and cannot change the model.
- Class weighting is used; synthetic oversampling is not used.
- The analysis unit remains the released recording because participant identity is unverified.

## Setup

### 1. Paths, dependencies and frozen benchmark design

In [ ]:
from pathlib import Path
import hashlib, json, platform, sys, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import signal
from IPython.display import display

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score,
    matthews_corrcoef, confusion_matrix, classification_report)
import joblib

PAPER_ROOT=Path(r'D:\EEG_AD_Research\paper1_experiments\paper1_outputs')
E07_ROOT=PAPER_ROOT/'p1_e07_1_full_cohort_reference_preprocessing_automated_qa'
OUTPUT_ROOT=PAPER_ROOT/'p1_e08_1_leakage_free_nonspiking_benchmark'
TABLES=OUTPUT_ROOT/'tables'; FIGURES=OUTPUT_ROOT/'figures'; MODELS=OUTPUT_ROOT/'models'; FEATURES=OUTPUT_ROOT/'features'
for path in (OUTPUT_ROOT,TABLES,FIGURES,MODELS,FEATURES): path.mkdir(parents=True,exist_ok=True)

E07_RUN_RECORD=E07_ROOT/'run_record.json'
RECORDING_MANIFEST=E07_ROOT/'tables'/'09_LOCKED_PREPROCESSING_ELIGIBLE_RECORDING_MANIFEST.csv'
EPOCH_MANIFEST=E07_ROOT/'tables'/'10_LOCKED_AUTOMATIC_QA_EPOCH_MANIFEST.csv'

SEED=20260828; FS=200.0; BOOTSTRAP_REPS=2000
CHANNELS=['Fp1','Fp2','F7','F3','Fz','F4','F8','T3','C3','Cz','C4','T4','T5','P3','Pz','P4','T6','O1','O2']
BANDS={'delta':(1,4),'theta':(4,8),'alpha':(8,13),'beta':(13,30),'low_gamma':(30,40)}
CLASS_ORDER=['Normal','MCI','AD']; SPLIT_ORDER=['train','validation','test']
TASKS={'three_class':['Normal','MCI','AD'],'normal_vs_ad':['Normal','AD'],'mci_vs_ad':['MCI','AD']}
REPRESENTATIONS=['spectral','temporal','hybrid']
MODEL_ORDER={'logistic':0,'linear_svm':1,'random_forest':2}
REP_ORDER={'spectral':0,'temporal':1,'hybrid':2}

plt.rcParams.update({'figure.dpi':120,'savefig.dpi':600,'font.size':9,
                     'axes.titlesize':10,'axes.labelsize':9,'pdf.fonttype':42,'ps.fonttype':42})
sns.set_theme(style='whitegrid',context='paper')

def sha256_file(path,block_size=4*1024*1024):
    digest=hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda:handle.read(block_size),b''): digest.update(block)
    return digest.hexdigest()
def save_table(frame,name):
    frame.to_csv(TABLES/f'{name}.csv',index=False)
    try: frame.to_excel(TABLES/f'{name}.xlsx',index=False)
    except Exception as exc: warnings.warn(f'Excel export skipped ({exc}); CSV was saved.')
def savefig(fig,name,width=8,height=4):
    fig.set_size_inches(width,height); fig.tight_layout()
    fig.savefig(FIGURES/f'{name}.png',dpi=600,bbox_inches='tight',facecolor='white')
    fig.savefig(FIGURES/f'{name}.pdf',bbox_inches='tight',facecolor='white')

print('Experiment: P1-E08.1')
print('Input:',E07_ROOT)
print('Output:',OUTPUT_ROOT)

## Data

### 2. Authenticate the completed P1-E07.1 lock

In [ ]:
required=[E07_RUN_RECORD,RECORDING_MANIFEST,EPOCH_MANIFEST]
missing=[str(path) for path in required if not path.is_file()]
if missing: raise FileNotFoundError('Required P1-E07.1 files are missing:\n'+'\n'.join(missing))

e07_run=json.loads(E07_RUN_RECORD.read_text(encoding='utf-8'))
recordings=pd.read_csv(RECORDING_MANIFEST,dtype={'recording_id':str,'serial':str})
epochs_manifest=pd.read_csv(EPOCH_MANIFEST,dtype={'recording_id':str,'serial':str})
for frame in (recordings,epochs_manifest):
    for column in ('recording_id','serial'):
        if column in frame: frame[column]=frame[column].astype(str).str.strip().str.zfill(5)

array_paths=recordings.array_path.map(Path)
array_exist=array_paths.map(Path.is_file)
array_hash_match=[]
for row in recordings[['array_path','array_sha256']].itertuples(index=False):
    array_hash_match.append(Path(row.array_path).is_file() and sha256_file(row.array_path)==row.array_sha256)

counts=(recordings.groupby(['official_split','experimental_label']).size().unstack(fill_value=0)
        .reindex(index=SPLIT_ORDER,columns=CLASS_ORDER,fill_value=0).astype(int))
input_checks=pd.DataFrame([
 {'check':'P1-E07.1 passed','pass':str(e07_run.get('status','')).startswith('P1-E07.1 PASSED'),'detail':e07_run.get('status','')},
 {'check':'recording manifest authenticated','pass':sha256_file(RECORDING_MANIFEST)==e07_run.get('locked_recording_manifest_sha256'),'detail':sha256_file(RECORDING_MANIFEST)},
 {'check':'epoch manifest authenticated','pass':sha256_file(EPOCH_MANIFEST)==e07_run.get('locked_epoch_manifest_sha256'),'detail':sha256_file(EPOCH_MANIFEST)},
 {'check':'914 recording rows','pass':len(recordings)==914 and recordings.recording_id.nunique()==914,'detail':len(recordings)},
 {'check':'10,821 selected epoch rows','pass':len(epochs_manifest)==10821,'detail':len(epochs_manifest)},
 {'check':'all epoch arrays exist','pass':array_exist.all(),'detail':int(array_exist.sum())},
 {'check':'all epoch-array hashes match','pass':all(array_hash_match),'detail':int(sum(array_hash_match))},
 {'check':'all classes remain in every split','pass':counts.gt(0).all().all(),'detail':counts.to_dict()},
 {'check':'analysis remains recording-level','pass':recordings.analysis_unit.astype(str).str.lower().eq('recording').all(),'detail':recordings.analysis_unit.value_counts().to_dict()},
 {'check':'participant identity remains unverified','pass':~recordings.participant_identity_verified.astype(str).str.lower().isin(['true','1']).any(),'detail':recordings.participant_identity_verified.value_counts(dropna=False).to_dict()},
])
save_table(input_checks,'01_p1_e07_1_input_authentication_checks'); display(counts); display(input_checks)
if not input_checks['pass'].all(): raise AssertionError('STOP — P1-E07.1 INPUT AUTHENTICATION FAILED')
print('P1-E07.1 INPUT AUTHENTICATED')

## Results

### 3. Extract diagnosis-independent epoch and recording features

In [ ]:
def integrate_psd(psd,freqs,low,high):
    mask=(freqs>=low)&(freqs<high if high<40 else freqs<=high)
    return np.trapezoid(psd[...,mask],freqs[mask],axis=-1)

def epoch_features(epochs):
    epochs=np.asarray(epochs,float)
    freqs,psd=signal.welch(epochs,fs=FS,nperseg=int(4*FS),axis=-1)
    total=integrate_psd(psd,freqs,1,40)+1e-18
    values=[]; names=[]; groups=[]
    for band,(low,high) in BANDS.items():
        power=integrate_psd(psd,freqs,low,high)
        for transform,prefix in [(np.log10(power+1e-18),f'logbp_{band}'),
                                 (power/total,f'relbp_{band}')]:
            values.append(transform)
            names.extend([f'{prefix}_{channel}' for channel in CHANNELS])
            groups.extend(['spectral']*len(CHANNELS))
    use=(freqs>=1)&(freqs<=40)
    probability=psd[...,use]/(psd[...,use].sum(axis=-1,keepdims=True)+1e-18)
    entropy=-(probability*np.log(probability+1e-18)).sum(axis=-1)/np.log(probability.shape[-1])
    cumulative=np.cumsum(psd[...,use],axis=-1)
    threshold=.95*cumulative[...,-1]
    edge_index=(cumulative>=threshold[...,None]).argmax(axis=-1)
    sef95=freqs[use][edge_index]
    alpha=(freqs>=8)&(freqs<=13)
    alpha_peak=freqs[alpha][np.argmax(psd[...,alpha],axis=-1)]
    for matrix,prefix in [(entropy,'spectral_entropy'),(sef95,'sef95_Hz'),(alpha_peak,'alpha_peak_Hz')]:
        values.append(matrix); names.extend([f'{prefix}_{channel}' for channel in CHANNELS]); groups.extend(['spectral']*len(CHANNELS))

    dx=np.diff(epochs,axis=-1); ddx=np.diff(dx,axis=-1)
    activity=np.var(epochs,axis=-1)
    mobility=np.sqrt(np.var(dx,axis=-1)/(activity+1e-18))
    mobility_dx=np.sqrt(np.var(ddx,axis=-1)/(np.var(dx,axis=-1)+1e-18))
    complexity=mobility_dx/(mobility+1e-18)
    line_length=np.mean(np.abs(dx),axis=-1)
    for matrix,prefix in [(np.log10(activity+1e-18),'hjorth_log_activity'),
                          (mobility,'hjorth_mobility'),(complexity,'hjorth_complexity'),
                          (line_length,'line_length')]:
        values.append(matrix); names.extend([f'{prefix}_{channel}' for channel in CHANNELS]); groups.extend(['temporal']*len(CHANNELS))
    matrix=np.concatenate(values,axis=1)
    return matrix,names,groups

blind_inputs=recordings[['recording_id','serial','array_path','array_sha256']].copy()
feature_rows=[]; feature_names=None; base_groups=None; extraction_audit=[]
for number,row in enumerate(blind_inputs.itertuples(index=False),1):
    try:
        if sha256_file(row.array_path)!=row.array_sha256: raise ValueError('array_hash_mismatch')
        with np.load(row.array_path,allow_pickle=False) as payload:
            epochs=np.asarray(payload['epochs_uv'],dtype=float)
            fs=float(payload['fs_Hz']); channels=payload['channels'].astype(str).tolist()
        if epochs.ndim!=3 or epochs.shape[1:]!=(19,2000): raise ValueError(f'unexpected_shape_{epochs.shape}')
        if not np.isclose(fs,FS) or channels!=CHANNELS: raise ValueError('sampling_rate_or_channel_order_mismatch')
        epoch_matrix,names,groups=epoch_features(epochs)
        if feature_names is None: feature_names=names; base_groups=groups
        elif names!=feature_names or groups!=base_groups: raise ValueError('feature_schema_changed')
        median=np.median(epoch_matrix,axis=0)
        iqr=np.quantile(epoch_matrix,.75,axis=0)-np.quantile(epoch_matrix,.25,axis=0)
        feature_rows.append(np.concatenate([median,iqr]))
        extraction_audit.append({'recording_id':row.recording_id,'serial':row.serial,
            'epochs':len(epochs),'feature_extraction_success':True,'error':''})
    except Exception as exc:
        extraction_audit.append({'recording_id':row.recording_id,'serial':row.serial,
            'epochs':0,'feature_extraction_success':False,'error':f'{type(exc).__name__}: {exc}'})
    if number%100==0 or number==len(blind_inputs): print(f'[{number:03d}/{len(blind_inputs)}] feature sets extracted')

extraction_audit=pd.DataFrame(extraction_audit)
if not extraction_audit.feature_extraction_success.all():
    save_table(extraction_audit,'02_feature_extraction_audit')
    raise RuntimeError('Feature extraction failed for one or more locked recordings.')

X=np.asarray(feature_rows,dtype='float32')
aggregate_names=[f'median__{name}' for name in feature_names]+[f'iqr__{name}' for name in feature_names]
aggregate_groups=base_groups+base_groups
feature_metadata=blind_inputs[['recording_id','serial']].copy()
catalog=pd.DataFrame({'feature_index':np.arange(len(aggregate_names)),'feature_name':aggregate_names,
                      'representation_group':aggregate_groups})
feature_npz=FEATURES/'02_LOCKED_RECORDING_LEVEL_NONSPIKING_FEATURE_MATRIX.npz'
np.savez_compressed(feature_npz,X=X,recording_id=feature_metadata.recording_id.to_numpy(dtype='U8'),
                    feature_names=np.asarray(aggregate_names,dtype='U96'),
                    representation_group=np.asarray(aggregate_groups,dtype='U16'))
save_table(extraction_audit,'02_feature_extraction_audit')
save_table(catalog,'03_feature_catalog')
save_table(feature_metadata,'04_diagnosis_independent_feature_row_manifest')
print('Feature matrix:',X.shape,'finite:',np.isfinite(X).all(),'SHA-256:',sha256_file(feature_npz))

### 4. Join labels after feature extraction and define official partitions

In [ ]:
metadata_columns=['recording_id','serial','official_split','experimental_label','age',
                  'analysis_unit','identity_group','participant_identity_verified','array_path','array_sha256']
metadata=feature_metadata.merge(recordings[metadata_columns],on=['recording_id','serial'],
                                how='left',validate='one_to_one')
if metadata[['official_split','experimental_label']].isna().any().any(): raise ValueError('Label/split join incomplete')
save_table(metadata,'05_LOCKED_RECORDING_FEATURE_METADATA_WITH_LABELS')

representation_indices={
 'spectral':np.flatnonzero(np.asarray(aggregate_groups)=='spectral'),
 'temporal':np.flatnonzero(np.asarray(aggregate_groups)=='temporal'),
 'hybrid':np.arange(X.shape[1])}
representation_summary=pd.DataFrame([{'representation':name,'features':len(indices)}
                                     for name,indices in representation_indices.items()])
save_table(representation_summary,'06_representation_dimensions'); display(representation_summary)
display(metadata.groupby(['official_split','experimental_label']).size().unstack(fill_value=0))

### 5. Training-only tuning and validation-based candidate selection

Every scaler is contained inside its estimator pipeline, so its mean and variance are learned only from the applicable training fold. The test split is not accessed in this cell.

In [ ]:
def model_specifications():
    return {
      'logistic':(Pipeline([('scale',StandardScaler()),('model',LogisticRegression(
          class_weight='balanced',max_iter=4000,random_state=SEED))]),
          {'model__C':[0.01,0.1,1.0,10.0]}),
      'linear_svm':(Pipeline([('scale',StandardScaler()),('model',LinearSVC(
          class_weight='balanced',dual='auto',max_iter=10000,random_state=SEED))]),
          {'model__C':[0.001,0.01,0.1,1.0]}),
      'random_forest':(Pipeline([('scale','passthrough'),('model',RandomForestClassifier(
          n_estimators=400,class_weight='balanced_subsample',n_jobs=-1,random_state=SEED))]),
          {'model__max_features':['sqrt',0.3],'model__min_samples_leaf':[1,3,5]})}

def core_metrics(y_true,y_pred):
    return {'accuracy':accuracy_score(y_true,y_pred),
            'balanced_accuracy':balanced_accuracy_score(y_true,y_pred),
            'macro_f1':f1_score(y_true,y_pred,average='macro'),
            'mcc':matthews_corrcoef(y_true,y_pred)}

candidate_rows=[]; selected_rows=[]; selected_objects={}
for task,classes in TASKS.items():
    task_mask=metadata.experimental_label.isin(classes).to_numpy()
    train_mask=task_mask & metadata.official_split.eq('train').to_numpy()
    validation_mask=task_mask & metadata.official_split.eq('validation').to_numpy()
    label_order=classes
    y=metadata.experimental_label.to_numpy()
    cv=StratifiedKFold(n_splits=5,shuffle=True,random_state=SEED)
    fitted_candidates={}
    for representation in REPRESENTATIONS:
        columns=representation_indices[representation]
        for model_name,(estimator,param_grid) in model_specifications().items():
            search=GridSearchCV(estimator,param_grid,scoring='balanced_accuracy',cv=cv,
                                n_jobs=-1,refit=True,return_train_score=False)
            search.fit(X[train_mask][:,columns],y[train_mask])
            validation_prediction=search.best_estimator_.predict(X[validation_mask][:,columns])
            metrics=core_metrics(y[validation_mask],validation_prediction)
            key=(representation,model_name)
            fitted_candidates[key]=search
            candidate_rows.append({'task':task,'representation':representation,'model':model_name,
                'n_train':int(train_mask.sum()),'n_validation':int(validation_mask.sum()),
                'features':len(columns),'cv_balanced_accuracy':float(search.best_score_),
                'best_params':json.dumps(search.best_params_,sort_keys=True),
                **{f'validation_{k}':float(v) for k,v in metrics.items()}})
    task_candidates=pd.DataFrame([row for row in candidate_rows if row['task']==task])
    task_candidates['model_order']=task_candidates.model.map(MODEL_ORDER)
    task_candidates['representation_order']=task_candidates.representation.map(REP_ORDER)
    winner=(task_candidates.sort_values(
        ['validation_balanced_accuracy','validation_macro_f1','cv_balanced_accuracy','model_order','representation_order'],
        ascending=[False,False,False,True,True]).iloc[0])
    selected_rows.append(winner.drop(labels=['model_order','representation_order']).to_dict())
    selected_objects[task]={'search':fitted_candidates[(winner.representation,winner.model)],
                            'representation':winner.representation,'model':winner.model,
                            'columns':representation_indices[winner.representation],
                            'classes':classes}
    print(task,'selected:',winner.representation,winner.model,'validation BA=',round(winner.validation_balanced_accuracy,4))

candidate_results=pd.DataFrame(candidate_rows)
selected_candidates=pd.DataFrame(selected_rows)
save_table(candidate_results,'07_training_cv_and_validation_candidate_results')
save_table(selected_candidates,'08_LOCKED_SELECTED_CANDIDATES_BEFORE_TEST')
display(selected_candidates[['task','representation','model','cv_balanced_accuracy',
                             'validation_balanced_accuracy','validation_macro_f1','best_params']])

### 6. Refit frozen candidates and evaluate the official test split once

In [ ]:
def bootstrap_intervals(y_true,y_pred,repetitions=BOOTSTRAP_REPS):
    y_true=np.asarray(y_true); y_pred=np.asarray(y_pred); rng=np.random.default_rng(SEED)
    rows=[]
    for _ in range(repetitions):
        index=rng.integers(0,len(y_true),len(y_true))
        if len(np.unique(y_true[index]))<len(np.unique(y_true)): continue
        rows.append(core_metrics(y_true[index],y_pred[index]))
    frame=pd.DataFrame(rows)
    return {metric:(float(frame[metric].quantile(.025)),float(frame[metric].quantile(.975)))
            for metric in frame.columns}

test_rows=[]; prediction_rows=[]; ci_rows=[]; reports=[]; importance_rows=[]
for task,locked in selected_objects.items():
    classes=locked['classes']; columns=locked['columns']; y=metadata.experimental_label.to_numpy()
    task_mask=metadata.experimental_label.isin(classes).to_numpy()
    development_mask=task_mask & metadata.official_split.isin(['train','validation']).to_numpy()
    test_mask=task_mask & metadata.official_split.eq('test').to_numpy()
    final_estimator=clone(locked['search'].best_estimator_)
    final_estimator.fit(X[development_mask][:,columns],y[development_mask])
    prediction=final_estimator.predict(X[test_mask][:,columns])
    metrics=core_metrics(y[test_mask],prediction)
    test_rows.append({'task':task,'representation':locked['representation'],'model':locked['model'],
                      'n_development':int(development_mask.sum()),'n_test':int(test_mask.sum()),
                      **{f'test_{k}':float(v) for k,v in metrics.items()}})
    test_meta=metadata.loc[test_mask,['recording_id','serial','official_split','experimental_label']].copy()
    test_meta['predicted_label']=prediction; test_meta['correct']=test_meta.experimental_label.eq(test_meta.predicted_label)
    test_meta['task']=task; prediction_rows.extend(test_meta.to_dict('records'))
    intervals_ci=bootstrap_intervals(y[test_mask],prediction)
    for metric,(low,high) in intervals_ci.items():
        ci_rows.append({'task':task,'metric':metric,'estimate':metrics[metric],
                        'ci95_low':low,'ci95_high':high,'bootstrap_repetitions':BOOTSTRAP_REPS})
    report=classification_report(y[test_mask],prediction,labels=classes,output_dict=True,zero_division=0)
    for label,values in report.items():
        if isinstance(values,dict): reports.append({'task':task,'label':label,**values})

    model=final_estimator.named_steps['model']
    if hasattr(model,'coef_'): importance=np.mean(np.abs(model.coef_),axis=0)
    elif hasattr(model,'feature_importances_'): importance=model.feature_importances_
    else: importance=np.full(len(columns),np.nan)
    chosen_names=np.asarray(aggregate_names)[columns]
    for rank,index in enumerate(np.argsort(importance)[::-1][:30],1):
        importance_rows.append({'task':task,'rank':rank,'feature':chosen_names[index],
                                'importance':float(importance[index]),'model':locked['model']})

    artifact={'estimator':final_estimator,'feature_indices':columns,'feature_names':chosen_names,
              'classes':classes,'representation':locked['representation'],'model_name':locked['model'],
              'source_feature_matrix_sha256':sha256_file(feature_npz)}
    joblib.dump(artifact,MODELS/f'{task}_LOCKED_NONSPIKING_BENCHMARK.joblib')

test_results=pd.DataFrame(test_rows); predictions=pd.DataFrame(prediction_rows)
confidence_intervals=pd.DataFrame(ci_rows); class_reports=pd.DataFrame(reports)
feature_importance=pd.DataFrame(importance_rows)
save_table(test_results,'09_FINAL_OFFICIAL_TEST_RESULTS')
save_table(confidence_intervals,'10_OFFICIAL_TEST_BOOTSTRAP_CONFIDENCE_INTERVALS')
save_table(predictions,'11_OFFICIAL_TEST_RECORDING_PREDICTIONS')
save_table(class_reports,'12_OFFICIAL_TEST_CLASSIFICATION_REPORTS')
save_table(feature_importance,'13_TOP_FEATURE_IMPORTANCE')
display(test_results); display(confidence_intervals)

### 7. Publication-ready benchmark figures

In [ ]:
fig,axes=plt.subplots(1,len(TASKS))
for ax,(task,classes) in zip(np.atleast_1d(axes),TASKS.items()):
    use=predictions.loc[predictions.task.eq(task)]
    matrix=confusion_matrix(use.experimental_label,use.predicted_label,labels=classes)
    sns.heatmap(matrix,annot=True,fmt='d',cmap='Blues',cbar=False,ax=ax,
                xticklabels=classes,yticklabels=classes)
    ax.set(title=task.replace('_',' '),xlabel='Predicted',ylabel='True')
savefig(fig,'Figure_01_official_test_confusion_matrices',12,3.5); plt.show()

plot=test_results.melt(id_vars=['task','representation','model'],
    value_vars=['test_balanced_accuracy','test_macro_f1','test_mcc'],
    var_name='metric',value_name='score')
plot['metric']=plot.metric.str.replace('test_','',regex=False).str.replace('_',' ')
fig,ax=plt.subplots(); sns.barplot(data=plot,x='task',y='score',hue='metric',ax=ax)
ax.set(ylim=(0,1),xlabel='',ylabel='Official test score',title='Locked conventional benchmark performance')
ax.tick_params(axis='x',rotation=20); ax.legend(frameon=False)
savefig(fig,'Figure_02_locked_benchmark_performance',8,4); plt.show()

fig,axes=plt.subplots(1,len(TASKS))
for ax,task in zip(np.atleast_1d(axes),TASKS):
    use=feature_importance.loc[feature_importance.task.eq(task)].head(12).sort_values('importance')
    ax.barh(use.feature,use.importance,color='#56B4E9'); ax.set_title(task.replace('_',' ')); ax.set_xlabel('Importance')
savefig(fig,'Figure_03_top_features_by_endpoint',14,5); plt.show()

## Checks

### 8. Final leakage and reproducibility gates

In [ ]:
model_files=list(MODELS.glob('*_LOCKED_NONSPIKING_BENCHMARK.joblib'))
checks=pd.DataFrame([
 {'check':'P1-E07.1 source authenticated','pass':input_checks['pass'].all(),'detail':'all source gates passed'},
 {'check':'one feature row per eligible recording','pass':X.shape[0]==len(recordings)==914 and feature_metadata.recording_id.nunique()==914,'detail':X.shape},
 {'check':'feature matrix entirely finite','pass':np.isfinite(X).all(),'detail':int((~np.isfinite(X)).sum())},
 {'check':'feature extraction diagnosis-independent','pass':'experimental_label' not in blind_inputs.columns and 'experimental_label' not in feature_metadata.columns,'detail':'labels joined after extraction'},
 {'check':'all three endpoint tasks completed','pass':set(test_results.task)==set(TASKS),'detail':test_results.task.tolist()},
 {'check':'all representation/model candidates evaluated','pass':len(candidate_results)==len(TASKS)*len(REPRESENTATIONS)*3,'detail':len(candidate_results)},
 {'check':'candidate selection completed before test evaluation','pass':len(selected_candidates)==len(TASKS),'detail':len(selected_candidates)},
 {'check':'official test predictions unique within task','pass':not predictions.duplicated(['task','recording_id']).any(),'detail':len(predictions)},
 {'check':'test predictions reconcile to reported test sizes','pass':predictions.groupby('task').size().to_dict()==test_results.set_index('task').n_test.astype(int).to_dict(),'detail':predictions.groupby('task').size().to_dict()},
 {'check':'all locked model artifacts saved','pass':len(model_files)==len(TASKS),'detail':[path.name for path in model_files]},
 {'check':'no synthetic oversampling used','pass':True,'detail':'class-weighted estimators only'},
 {'check':'no Riemannian representation used','pass':True,'detail':'spectral, temporal and hybrid conventional features only'},
 {'check':'analysis remains recording-level','pass':metadata.analysis_unit.astype(str).str.lower().eq('recording').all(),'detail':metadata.analysis_unit.value_counts().to_dict()},
 {'check':'participant identity remains unverified','pass':~metadata.participant_identity_verified.astype(str).str.lower().isin(['true','1']).any(),'detail':metadata.participant_identity_verified.value_counts(dropna=False).to_dict()},
])
save_table(checks,'14_final_integrity_and_leakage_checks'); display(checks)
status=('P1-E08.1 PASSED — LEAKAGE-FREE NON-SPIKING BENCHMARK LOCKED'
        if checks['pass'].all() else 'STOP — P1-E08.1 CHECKS FAILED; DO NOT BEGIN BRAIN-INSPIRED MODELING')

feature_metadata_csv=TABLES/'05_LOCKED_RECORDING_FEATURE_METADATA_WITH_LABELS.csv'
selected_csv=TABLES/'08_LOCKED_SELECTED_CANDIDATES_BEFORE_TEST.csv'
test_csv=TABLES/'09_FINAL_OFFICIAL_TEST_RESULTS.csv'
run_record={'experiment':'P1-E08.1 leakage-free CAUEEG non-spiking representation and benchmark lock',
 'timestamp_utc':pd.Timestamp.utcnow().isoformat(),'status':status,
 'source_p1_e07_run_record':str(E07_RUN_RECORD),'source_recording_manifest_sha256':sha256_file(RECORDING_MANIFEST),
 'source_epoch_manifest_sha256':sha256_file(EPOCH_MANIFEST),'recordings':len(recordings),'epochs':len(epochs_manifest),
 'feature_matrix':str(feature_npz),'feature_matrix_sha256':sha256_file(feature_npz),
 'feature_metadata_csv':str(feature_metadata_csv),'feature_metadata_sha256':sha256_file(feature_metadata_csv),
 'selected_candidates_csv':str(selected_csv),'selected_candidates_sha256':sha256_file(selected_csv),
 'official_test_results_csv':str(test_csv),'official_test_results_sha256':sha256_file(test_csv),
 'tasks':TASKS,'representations':REPRESENTATIONS,'model_families':list(model_specifications()),
 'selection_protocol':'5-fold training CV -> official validation candidate selection -> train+validation refit -> one official test evaluation',
 'synthetic_oversampling_used':False,'riemannian_methods_used':False,
 'identity_interpretation':{'analysis_unit':'recording','participant_identity_verified':False,
                            'official_no_overlap_split_preserved':True},
 'python':sys.version,'platform':platform.platform()}
(OUTPUT_ROOT/'run_record.json').write_text(json.dumps(run_record,indent=2),encoding='utf-8')
print(json.dumps(run_record,indent=2))
if not checks['pass'].all(): raise AssertionError(status)

## Next Steps

After a complete P1-E08.1 PASS:

1. Treat the locked conventional test results as the baseline—not as a source for further tuning.
2. Proceed to the brain-inspired representation stage using the same P1-E07.1 arrays, recording-level identity and official splits.
3. Fit spike-encoding normalization, STDP parameters and any learned representation using training data only.
4. Use validation data for selection and the test split once for the frozen final comparison.
5. External datasets with verified participant IDs can reuse the feature extractor, but all grouping and resampling must then operate at participant level.

In [ ]:
from pathlib import Path
import pandas as pd
import json
import hashlib

# Original executed P1-E08.1 output directory
OUTPUT_ROOT = Path(
    r"D:\EEG_AD_Research\paper1_experiments\paper1_outputs"
    r"\p1_e08_1_leakage_free_nonspiking_benchmark"
)

TABLES = OUTPUT_ROOT / "tables"
MODELS = OUTPUT_ROOT / "models"
SEED = 20260828

SELECTED_CSV = (
    TABLES / "08_LOCKED_SELECTED_CANDIDATES_BEFORE_TEST.csv"
)

EXPECTED_SHA256 = (
    "8526b1c5e0577f9184c91fb7efe86c5f2d8bc4602efe41202f28d93adc3f47f4"
)

def file_sha256(path):
    digest = hashlib.sha256()

    with Path(path).open("rb") as handle:
        for block in iter(
            lambda: handle.read(4 * 1024 * 1024), b""
        ):
            digest.update(block)

    return digest.hexdigest()


# ---------------------------------------------------------
# Check that the original output directory is available
# ---------------------------------------------------------
if not OUTPUT_ROOT.is_dir():
    raise FileNotFoundError(
        "P1-E08.1 output directory was not found:\n"
        f"{OUTPUT_ROOT}\n\n"
        "Check the path before continuing."
    )

if not SELECTED_CSV.is_file():
    raise FileNotFoundError(
        "The locked selected-candidate table was not found:\n"
        f"{SELECTED_CSV}"
    )


# ---------------------------------------------------------
# Authenticate the locked table
# ---------------------------------------------------------
observed_hash = file_sha256(SELECTED_CSV)

print("Expected SHA-256:", EXPECTED_SHA256)
print("Observed SHA-256:", observed_hash)

if observed_hash != EXPECTED_SHA256:
    raise AssertionError(
        "Authentication failed. Do not use this table until the "
        "difference has been investigated."
    )

print("LOCKED SELECTED-CANDIDATE TABLE AUTHENTICATED")


# ---------------------------------------------------------
# Recover the full, untruncated parameter dictionaries
# ---------------------------------------------------------
selected = pd.read_csv(SELECTED_CSV)

required_columns = {
    "task",
    "representation",
    "model",
    "best_params"
}

missing_columns = required_columns - set(selected.columns)

if missing_columns:
    raise KeyError(
        f"Missing required columns: {sorted(missing_columns)}"
    )

selected["decoded_params"] = selected["best_params"].apply(
    json.loads
)

exact_rows = []

for row in selected.itertuples(index=False):
    params = row.decoded_params

    exact_rows.append({
        "endpoint": row.task,
        "representation": row.representation,
        "classifier": row.model,
        "C": params.get("model__C"),
        "n_estimators": (
            400 if row.model == "random_forest" else None
        ),
        "max_features": params.get("model__max_features"),
        "min_samples_leaf": params.get(
            "model__min_samples_leaf"
        ),
        "class_weight": (
            "balanced_subsample"
            if row.model == "random_forest"
            else "balanced"
        ),
        "random_state": SEED
    })

exact_settings = pd.DataFrame(exact_rows)

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

display(exact_settings)


# ---------------------------------------------------------
# Save the exact manuscript-ready parameter record
# ---------------------------------------------------------
EXACT_OUTPUT = (
    TABLES / "15_EXACT_SELECTED_MODEL_HYPERPARAMETERS.csv"
)

exact_settings.to_csv(EXACT_OUTPUT, index=False)

print("\nAuthenticated source:", SELECTED_CSV)
print("SHA-256:", observed_hash)
print("Exact parameter record saved:", EXACT_OUTPUT)

In [ ]:
verification_rows = []

for model_path in sorted(
    MODELS.glob("*_LOCKED_NONSPIKING_BENCHMARK.joblib")
):
    artifact = joblib.load(model_path)
    estimator = artifact["estimator"]
    params = estimator.get_params()

    verification_rows.append({
        "artifact": model_path.name,
        "endpoint": model_path.name.split(
            "_LOCKED_NONSPIKING_BENCHMARK"
        )[0],
        "representation": artifact["representation"],
        "classifier": artifact["model_name"],
        "C": params.get("model__C"),
        "n_estimators": params.get("model__n_estimators"),
        "max_features": params.get("model__max_features"),
        "min_samples_leaf": params.get(
            "model__min_samples_leaf"
        ),
        "class_weight": params.get("model__class_weight"),
        "random_state": params.get("model__random_state")
    })

artifact_verification = pd.DataFrame(verification_rows)
display(artifact_verification)

artifact_verification.to_csv(
    OUTPUT_ROOT / "tables" /
    "16_SERIALIZED_MODEL_PARAMETER_VERIFICATION.csv",
    index=False
)